<a href="https://colab.research.google.com/github/dom-dang/7.C01/blob/main/7_C01_PSET3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  <center> Problem Set 3 <center>
<center> Spring 2026 <center>
<center> 3.C01/3.C51, 7.C01/7.C51, 10.C01/10.C51, 20.C01/20.C51 <center>
<center> Due: Monday, April 22, 2026 at 3:00 PM ET. <center>

<b>Name:</b> Dominique Dang

<b>Kerberos id:</b> ddang

## Learning Objective
In this problem set, you will apply Graph Neural Networks (GNNs) to predict Synthetic Lethality (SL) in cancer. You will:
- Construct and analyze Synthetic Lethality graphs and Knowledge Graphs (KGs) from biological data.
- Understand the role of sparsity and graph structure in biological modeling.
- Implement, train, and evaluate a GNN model (including Graph Attention/Convolutional layers) using PyTorch and DGL.
- Analyze model performance using metrics like AUC, Precision, Recall, and F1 score.
- Interpret model predictions in the context of cancer biology (e.g., KRAS/HOXC11 interactions).


## Instructions
- This problem set has two modeling tasks with several sub-questions. Some are marked grad version, which are required for graduate students (X.C51) but optional for others. Points for all students are in <span style="color:blue">blue</span>, while grad-only points are in <span style="color:orange">orange</span>. There is one problem that is undergrad only in <span style="color:purple">purple</span>. The total points are 75 for undergraduates and 100 for graduates.
- To get started, make your own copy of this notebook template in Colab (e.g., “Save a copy in Drive”) before editing.
    - Important: this problem set requires a GPU. In Google Colab go to `Edit -> Notebook settings` and set the `Hardware accelerator` to a GPU before running the notebook (changing the runtime resets the notebook). See the GPU section below for additional help.
- Collaboration is encouraged and AI tools are permitted, but submitting work that is not your own is plagiarism. Any collaboration or assistance from others or from an LLM (including utilities integrated in Colab) must be described at the end of your submission.
- Additional notes about how to use this template:
    - Put your code in the code blocks flagged with `############# Code ##########`.
    -  Numerical answers yielded from running the code should be included in an Answer Block (see next cell).
    - We have provided print statements where numerical answers are expected.
    -  Your answer should be contained in a variable which you defined either in the Answer Block or the Code Block.
    - When a qualitative answer is expected, place those comments as Markdown/Text cells; when asked for within Code blocks, you can write answer as code comments by placing a # before your answer.

- Submission: upload your completed `pset1.ipynb` to Gradescope. Ensure the notebook runs without error and includes all necessary code, plots, and outputs. Comments are encouraged; place conceptual answers in Markdown/Text cells.


## Background
- Two genes are **synthetic lethal (SL)** if mutations in both of them reduce cell viability and/or lead to cell death, but mutations in either of them individually have minimal effects.

- SL is particularly relevant in the context of human cancers, and identifying SL gene pairs is essential for developing **anti-cancer therapeutics and drugs.**

- There are many **biological factors (a.k.a. SL-related factors)** that can explain why a gene pair is synthetic lethal, including whether or not the two genes interact or share a molecular function, among many others. There are databases with annotated and curated information about the nature of the interaction between approximately 10,000 human genes, namely SL relationships, such as the [SynLethDB Database](https://synlethdb.sist.shanghaitech.edu.cn/#/search).

- In this exercise, you will leverage published work [Li et al.](https://academic.oup.com/bioinformatics/article/39/2/btad015/6988048) to **train a Graph Neural Network (GNN) for prediction of SL interactions.** In the GNN field, this is known as "link prediction". This is a **supervised classification problem** where either a gene pair is synthetic lethal (SL) or not, and you will use a GNN to make predictions on human genes.

## 0. Load relevant packages (~2 min)

Includes installations of relevant packages & mounting Google Drive.
A new window will pop up for you to authorize access to your Google Drive - please accept the two requests you will be presented with.


In [ ]:
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.4.0+cu121.html

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
mydrive = '/content/drive/MyDrive/'

In [ ]:
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html

In [ ]:
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.4.0+cu121.html

In [ ]:
import os
from os.path import join
import sys
import random
from random import sample
import json
import io
import requests
import collections
from time import time
from code import interact
from traceback import print_tb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils import data
from torch.utils.data import Dataset, DataLoader, Subset
from torch_scatter import scatter_mean

import scipy.sparse as sp
from scipy.sparse import csr_matrix

import networkx as nx

import dgl
import dgl.function as fn
from dgl.nn import GATConv

In [ ]:
#setting random see for reproducibility
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
#check if GPU is actually available
torch.cuda.is_available()

# Part 1: Constructing Synthetic Lethality and Knowledge Graphs (60 points)

When predicting SL using a GNN, there are two complementary ways of representing data (i.e., two types of graphs - **SL and Knowledge graphs**), and both graphs are input to the model.

1. Modeling SL interactions between gene pairs using a **SL graph** $S = (V, E)$, where nodes $V$ correspond to a set of $n$ genes and edges $E$ correspond to a SL relationship. If there is an edge between two genes, the value of the adjacency matrix $A_{1,2}$ ($\mathbb{R}^{n \times n}$) between two genes $v_1$ and $v_2$ is 1; otherwise, it is 0.


<div align="center">
  <img src="https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps3-SLGNN/figures/download.png" width="500px" />
</div>
<div align="center">Synthetic Lethality graph (<a href="https://academic.oup.com/bioinformatics/article/39/2/btad015/6988048">image</a>)</div>


2. Modeling the relationships between genes in a **Knowledge Graph (KG)** $G = (V_e, E_r)$, where nodes $V_e$ correspond to a set of gene entities $e$ and edges $E_r$ correspond to a set of SL-related factors/relationships $r$. Understanding what biological factors cause SL is a key part of predicting SL (Table 2 in [Li et al.](https://academic.oup.com/bioinformatics/article/39/2/btad015/6988048)). Each KG can be represented as $G = \{(h,t,r) | h,t \in V_e , r \in G_r\}$. The $(h,t,r)$ triple refers to head (equivalent to nodes in the SL graph) and tail (equivalent to edges in the SL graph) entities, and relationships between heads and tails, respectively.

<div align="center">
  <img src="https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps3-SLGNN/figures/download-2.png" width="400px" />
</div>
<div align="center">Knowledge Graph (<a href="https://academic.oup.com/bioinformatics/article/39/2/btad015/6988048">image</a>)</div>

The questions in Part 1 require a mixture of coding - see instructions in the relevant Colab notebook - as well as answering the following questions in markdown cells:

## 1.1 <span style="color:blue">(5 points) </span> Get Data to construct a Knowledge Graph (KG)

**Run the code cells below and answer the questions that follow.**

In [ ]:
torch.set_default_dtype(torch.float32)
NEIGHBOR_NUM = 16

In [ ]:
class KGDataset(Dataset):
    def __init__(self):
        kg_dict = np.load('kg_dict.npy', allow_pickle=True).item()
        hs = np.load('hs.npy', allow_pickle=True)
        rs = np.load('rs.npy', allow_pickle=True)
        ts = np.load('ts.npy', allow_pickle=True)

        self.kg_dict = kg_dict
        self.heads = hs
        self.rels = rs
        self.tails = ts
        self.graph = None
        self.sampled_edges, self.sampled_rels, self.neighbor_num = self.get_Neighborhood()

    @property
    def entity_count(self):
        return max(max(self.tails), max(self.heads)) + 2

    @property
    def relation_count(self):
        return max(self.rels) + 2

    def __len__(self):
        return len(self.kg_dict)

    def __getitem__(self, index):
        head = self.heads[index]
        relation, pos_tail = random.choice(self.kg_dict[head])
        while True:
            neg_head = random.choice(self.heads)
            neg_tail = random.choice(self.kg_dict[neg_head])[1]
            if (relation, neg_tail) in self.kg_dict[head]:
                continue
            else:
                break
        return head, relation, pos_tail, neg_tail

    def get_Neighborhood(self):
        print('sampling neighborhood...')
        neighbor_num = NEIGHBOR_NUM
        edges = np.load('sample_edges.npy', allow_pickle=True).item()
        rels = np.load('sample_rels.npy', allow_pickle=True).item()
        print('successfully loaded...')
        return edges, rels, neighbor_num

In [ ]:
kg_dataset = KGDataset()
heads = kg_dataset.heads   #head
relations = kg_dataset.rels    #relationship
tails = kg_dataset.tails   #tail

**Question**: For each gene entity in the KG, 16 neighbors are being sampled in the code provided in the cells above (you should simply run them), which will be used during training. How could changing the number of sampled neighbors impact model performance?

**Write answer here**

## 1.2 <span style="color:blue">(5 points) </span> Get Data to construct a SL graph & select gene pairs for training and validation


Run the code cells below and answer the following question. You will need to write some lines of code to answer Q1.


In [ ]:
class SLDataset(Dataset):
    def __init__(self, fold_n) -> None:
        super().__init__()
        print(f'loading sl dataset...')
        self.fold_n = fold_n
        self.sl_data = np.load('/content/drive/MyDrive/sl2id.npy', allow_pickle=True)
        self.df = pd.DataFrame(self.sl_data, columns=['gene_a', 'gene_b', 'label'])
        self.df = self.df[self.df['label'] == 1][['gene_a', 'gene_b']]
        self.df_reverse = pd.DataFrame()
        self.df_reverse['gene_a'] = self.df['gene_b']
        self.df_reverse['gene_b'] = self.df['gene_a']
        self.df_dataset = pd.concat([self.df, self.df_reverse])
        self.reindex_dict = self.get_reindex_dict()
        self.an_reindex_dict = {y: x for x, y in self.reindex_dict.items()}
        self.train_df, self.val_df = self.get_df()
        self.train_a, self.train_b, self.train_label, self.val_a, self.val_b, self.val_label = self.get_data()
        self.SLGraph = self.getSLGraph()

    @property
    def n_gene(self):
        return max(self.reindex_dict.values()) + 1

    @property
    def trainDataSize(self):
        return len(self.train_a)

    @property
    def valDataSize(self):
        return len(self.val_a)

    def get_df(self):
        train_df = pd.read_csv('/content/drive/MyDrive/train.txt', sep='\t')
        val_df = pd.read_csv('/content/drive/MyDrive/val.txt', sep='\t')
        train_df = self.reindex_sl(train_df)
        val_df = self.reindex_sl(val_df)
        return train_df, val_df

    def get_data(self):
        train_a = np.array(self.train_df['gene_a'])
        train_b = np.array(self.train_df['gene_b'])
        train_label = np.array(self.train_df['label'])
        val_a = np.array(self.val_df['gene_a'])
        val_b = np.array(self.val_df['gene_b'])
        val_label = np.array(self.val_df['label'])
        return train_a, train_b, train_label, val_a, val_b, val_label

    def getSLGraph(self):
        gene_a = np.array(self.train_df[self.train_df['label'] == 1]['gene_a'])
        gene_b = np.array(self.train_df[self.train_df['label'] == 1]['gene_b'])
        SLGraph = csr_matrix((np.ones(len(gene_a)), (gene_a, gene_b)), shape=(self.n_gene, self.n_gene))
        SLGraph += sp.eye(SLGraph.shape[0], format='csr')
        return SLGraph

    def getPosNeighbors(self, ids):
        posNeighbors = []
        for id in ids:
            posNeighbors.append(self.SLGraph[id].nonzero()[1])
        return posNeighbors

    def get_reindex_dict(self):
        df = self.df_dataset
        set_a = set(list(df['gene_a']))
        set_b = set(list(df['gene_b']))
        set_all = sorted(list(set_a | set_b))
        reindex_dict = dict()
        for i in range(len(set_all)):
            reindex_dict[set_all[i]] = i
        return reindex_dict

    def reindex_sl(self, df_for_reidx):
        reindex_dict = self.reindex_dict
        df = df_for_reidx.copy()
        df['gene_a'] = df['gene_a'].apply(lambda x: reindex_dict.get(x))
        df['gene_b'] = df['gene_b'].apply(lambda x: reindex_dict.get(x))
        return df

In [ ]:
sl_dataset = SLDataset(fold_n=1) #just training model on 1 fold of data (no CV)

Print the sizes of the training and validation datasets.




In [ ]:
############ CODE ############

print(train_data_size)
print(val_data_size)
############ CODE ############

**Question**:  How many gene pairs are in the training and validation sets? How do these numbers compare to the total number of human genes included in the study (~10,000)?
  

**Write answer here**

## 1.3 <span style="color:blue">(25 points) </span> Construct SL graph

In [ ]:
from scipy.sparse import find

sparse_matrix = sl_dataset.SLGraph
rows, cols, values = find(sparse_matrix)

subset_size = int(len(set(rows)) * 0.05)  # Plotting only a subset of genes for visual clarity
subset_rows = random.sample(sorted(set(rows)), subset_size)
subset_cols = random.sample(sorted(set(cols)), subset_size)

edges = [(row, col) for row, col in zip(rows, cols) if row in subset_rows and col in subset_cols]

G = nx.DiGraph()
G.add_edges_from(edges)
G.remove_edges_from(list(nx.selfloop_edges(G)))

pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=True, node_size=50, node_color='pink', edge_color='gray')

plt.title('SL Graph')

In [ ]:
dense_matrix = sparse_matrix.toarray()
np.fill_diagonal(dense_matrix, 0)
binary_matrix = (dense_matrix != 0).astype(int)
print(binary_matrix.shape)
#plotting only some genes to better visualize patterns
#can play around with what genes are shown
plt.imshow(binary_matrix[6200:6350, 6200:6350], cmap='binary')
plt.colorbar()
plt.title('Gene-gene interaction matrix')
plt.xlabel('Gene a')
plt.ylabel('Gene b')

**Question 1**:   What does the SL graph represent? What are its nodes and edges? (**5 points**)


**Write answer here**

**Question 2**:  What are the key differences between applying dropout to nodes vs. edges during training, and how might each affect model learning and performance? (**5 points**)
  

**Write answer here**

**Question 3**:  In the context of the SL graph, what are the dimensions of the binary matrix and do they make sense to you? In the context of the problem, comment on patterns you see in the matrix, namely the almost perfect yet incomplete symmetry across the diagonal as well as the sparsity of the matrix. Does its diagonal look reasonable to you?  (**10 points**)
  

**Write answer here**

**Question 4**: Considering what you have learned in class, comment on the benefits and dangers of sparsity in the context of GNNs applied to biological questions, namely in the context of predicting SL gene pairs. (**5 points**)
  

**Write answer here**

## 1.4 <span style="color:blue">(25 points) </span> Load KG Data to Construct Knowledge Graph

In [ ]:
def load_data(inverse_r):
    url = "https://github.com/zy972014452/SLGNN/raw/main/data/SL/raw/kg2id.txt"
    response = requests.get(url)
    content = response.text
    kg2id_np = np.loadtxt(content.splitlines(), dtype=np.int64)

    max = 0
    r_max = 0
    for i in range(len(kg2id_np)):
        h = kg2id_np[i][0]
        t = kg2id_np[i][2]
        r = kg2id_np[i][1]
        if h > max:
            max = h
        if t > max:
            max = t
        if r > r_max:
            r_max = r
    n_entities = max + 1
    n_relations = r_max + 1

    graph = nx.MultiDiGraph()
    for h, r, t in kg2id_np:
        graph.add_edge(h, t, key=r)
        if inverse_r == True:
            graph.add_edge(t, h, key=r + n_relations)

    return graph, n_entities, n_relations

**Task**: load the KG graph and print the number of nodes, edges, entities, and relations

In [ ]:
############ CODE ############


############ CODE ############

In [ ]:
# Plot the graph you loaded - plotting only 1% of data for visual clarity
nodes_subset = np.random.choice(kg_graph.nodes(), size=int(0.01 * len(kg_graph)), replace=False)
sub_graph = nx.MultiDiGraph()
sub_graph.add_nodes_from(nodes_subset)
for node in nodes_subset:
    edges = kg_graph.edges(node, keys=True, data=True)
    for edge in edges:
        if edge[1] in nodes_subset:
            sub_graph.add_edge(edge[0], edge[1], key=edge[2])

pos = nx.spring_layout(sub_graph)

# excluding nodes that are not connected by edges
sub_graph_all = sub_graph.to_undirected()
components = list(nx.connected_components(sub_graph_all))
largest_component = max(components, key=len)
final_sub_graph = sub_graph.subgraph(largest_component)
nx.draw(final_sub_graph, pos, with_labels=True, node_size=50, node_color='skyblue',  edge_color='gray', font_size=8)
plt.title("Knowledge Graph")

**Question 1**: How many nodes and edges does the KG have? And how many entities and relations are defined in the KG object? (**5 points**) Additional coding question worth **5 points**.
  

**Write answer here**

**Question 2**: What does the boolean variable `inverse_r` mean? How does changing it from `False` to `True` change the appearance of the KG, and the total number of edges and nodes in the KG? (**5 points**)
  

**Write answer here**

**Question 3**: What is the rationale behind integrating these two graph types - SL graph and KG - and how might that improve model performance? Are there any downsides of weighting one type of graph more heavily relative to the other? (**10 points**)

**Write answer here**

# Part 2: Training a GNN to predict SL gene pairs (30 points)

## Background on GNNs
- Each layer in a GNN operates on graph-structured data in two steps: a **message** step and an **update** step. In this example, the message step takes information from connected nodes in the KG, and the update function transforms these parameterized messages to update the features (gene and entity embeddings) learned for each node in both the KG and SL graphs. Information from the most relevant neighboring nodes need to be **aggregated** during message passing. Aggregated information comes from neighboring nodes and is performed over multiple iterations of message passing (a.k.a hops).

- After multiple convolutional layers, each node receives a parameterized embedding. The number of layers in the neural network (NN) is chosen based on the dimensions of the graph (Figure 2 - Knowledge Graph, above). A larger number of layers allows getting information about entities that are farther away and potentially improve model performance (as long as they do not exceed the size of the graph). Complex operations are required to map all the node embeddings onto a predicted scalar value. The most important function is the **softmax function** that takes this predicted scalar value and calculates the **probability of gene pair being SL.** Information from the KG is hence used for predicting edges in the SL graph.

---
**Key aspects of the model:**

1. **Factor modeling**: Modeling different SL-related factors to reflect how they can result in an SL interaction. Factor embeddings are based on the biological relationships defined in KG (Figure 2).
2. Genes represented in the SL graph are input to the **Knowledge Graph Message Aggregation Graph Convolutional Network (GCN)**, consisting of multiple layers that learn embeddings for a gene at the entity level, leveraging information from the KG.
3. The **Factor-based Message Aggregation Graph Attention Network (GAN)** updates these embeddings to generate new ones using information about preferences for different SL-related factors encoded in the KG.


- In both cases, embeddings are created based on features learned from neighboring nodes (hence the relevance of a graph structure!), with **each neighbor contributing differently to a relationship**. These embeddings are then used to make link predictions on SL graphs (i.e., is a gene pair SL?) (Figure 3).

- Given how in the KG, **two entities can be related through different relationships**, this model leverages attention mechanisms to understand which factors are the most relevant for a given SL interaction. Attention will be covered in detail in future lectures. All the information/context required to complete this PSET is provided here and in the questions below.
  
<div align="center">
  <img src="https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps3-SLGNN/figures/download-3.png" width="700px" />
</div>
<div align="center">Figure 3: Overview of GNN to predict SL (<a href="https://academic.oup.com/bioinformatics/article/39/2/btad015/6988048">image</a>)</div>

## 2.1 <span style="color:blue">(5 points) </span> Create Datasets and Dataloaders for Training & Validation

You will start by constructing your dataset and loader, with `torch.utils.data.Dataset` and `torch.utils.data.DataLoader`.


In [ ]:
#Define Model Hyperparameters

class ArgsConfig:
    def __init__(self):

        #training parameters
        self.epoch = 60
        self.batch_size = 5000
        self.dim = 64
        self.lr = 3e-3

        #regularization parameters (prevent overfitting)
        self.l2 = 1e-4
        self.sim_regularity = 1e-3
        self.node_dropout = True
        self.node_dropout_rate = 0.5
        self.mess_dropout = True
        self.mess_dropout_rate = 0.1

        #other relevant parameters for the GNN
        self.context_hops = 3
        self.n_factors = 4
        self.ind = 'distance'

        self.cuda = True #Change depending on running on GPU/CPU
        self.gpu_id = 0
        self.fold_n = 1
        self.inverse_r = False


args_config = ArgsConfig()

In [ ]:
class KGCNDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        gene_a = np.array(self.df.iloc[idx]['gene_a'])
        gene_b = np.array(self.df.iloc[idx]['gene_b'])
        label = np.array(self.df.iloc[idx]['label'], dtype=np.float32)
        return gene_a, gene_b, label

**Task**: Construct train and validation data loader with using the train and validation datasets. Note the relevant hyperparameters have been defined in the **class** `ArgsConfig`.

In [ ]:
#in this line of code, call the training set defined in the KGCNDataset
train_dataset =
#in this line of code create, create the DataLoader for the training set
train_loader =
#in this line of code, call the validation set defined in the KGCNDataset
val_dataset =
#in this line of code create, create the DataLoader for the validation set
val_loader =

## 2.2 <span style="color:blue">(7.5 points) </span> : Model Architecture

This model includes 3 classes: **Graph Attention Network (GAT) Class**, **Aggregator Class**, and **GraphConv Class**. The final model that calls all classes is named **SLModel.**

> **Note**:  Despite the extensive code shown below, you are not expected to understand every single line of it. The questions in each section are meant to guide your understanding of the code. Our emphasis is on your understanding of why and how GNNs are useful in the context of predicting SL.


**Run the code cells below. This question requires no coding, just answer the questions below.**

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def distance_correlation(tensor_1, tensor_2):
    """
    Compute the distance correlation between two 1D tensors.
    Reference: https://en.wikipedia.org/wiki/Distance_correlation
    """
    channel = tensor_1.shape[0]
    device = tensor_1.device
    zeros = torch.zeros(channel, channel, device=device)
    zero = torch.zeros(1, device=device)

    tensor_1, tensor_2 = tensor_1.unsqueeze(-1), tensor_2.unsqueeze(-1)

    a_, b_ = torch.matmul(tensor_1, tensor_1.t()) * 2, torch.matmul(tensor_2, tensor_2.t()) * 2
    tensor_1_square, tensor_2_square = tensor_1**2, tensor_2**2

    a = torch.sqrt(torch.max(tensor_1_square - a_ + tensor_1_square.t(), zeros) + 1e-8)
    b = torch.sqrt(torch.max(tensor_2_square - b_ + tensor_2_square.t(), zeros) + 1e-8)

    A = a - a.mean(dim=0, keepdim=True) - a.mean(dim=1, keepdim=True) + a.mean()
    B = b - b.mean(dim=0, keepdim=True) - b.mean(dim=1, keepdim=True) + b.mean()

    dcov_AB = torch.sqrt(torch.max((A * B).sum() / channel**2, zero) + 1e-8)
    dcov_AA = torch.sqrt(torch.max((A * A).sum() / channel**2, zero) + 1e-8)
    dcov_BB = torch.sqrt(torch.max((B * B).sum() / channel**2, zero) + 1e-8)

    return dcov_AB / torch.sqrt(dcov_AA * dcov_BB + 1e-8)

### Graph Attention Network (GAT) Class

In [ ]:
class GAT(nn.Module):
    def __init__(self, num_layers, in_dim, num_hidden, num_classes, heads,
                 activation, feat_drop, attn_drop, negative_slope, residual):
        super(GAT, self).__init__()
        self.num_layers = num_layers
        self.activation = activation
        self.gat_layers = nn.ModuleList()

        # Input projection (no residual)
        self.gat_layers.append(
            GATConv(in_dim, num_hidden, heads[0], feat_drop, attn_drop,
                    negative_slope, False, activation))

        # Hidden layers
        for l in range(1, num_layers):
            self.gat_layers.append(
                GATConv(num_hidden * heads[l - 1], num_hidden, heads[l],
                        feat_drop, attn_drop, negative_slope, residual,
                        activation))

        # Output projection
        self.gat_layers.append(
            GATConv(num_hidden * heads[-2], num_classes, heads[-1], feat_drop,
                    attn_drop, negative_slope, residual, None))

    def forward(self, g, inputs):
        h = inputs
        for l in range(self.num_layers):
            h = self.gat_layers[l](g, h).flatten(1)
        return self.gat_layers[-1](g, h).mean(1)

### Aggregator Class


In [ ]:
class Aggregator(nn.Module):
    """
    Relational Path-aware Convolution Network
    """
    def __init__(self, n_genes, n_factors, reindex_dict):
        super(Aggregator, self).__init__()
        self.n_genes = n_genes
        self.n_factors = n_factors
        self.reindex_dict = reindex_dict

    def forward(self, GATConv, sl_graph, entity_emb, gene_sl_emb, latent_emb,
                edge_index, edge_type, interact_mat, weight, disen_weight_att):

        n_entities = entity_emb.shape[0]
        channel = entity_emb.shape[1]

        # KG aggregate
        head, tail = edge_index
        edge_relation_emb = weight[edge_type]
        neigh_relation_emb = entity_emb[tail] * edge_relation_emb
        entity_agg = scatter_mean(neigh_relation_emb, index=head, dim=0, dim_size=n_entities)

        latent_emb = torch.mm(F.softmax(disen_weight_att, dim=-1), weight)
        score_ = torch.mm(gene_sl_emb, latent_emb.t())
        score = F.softmax(score_, dim=1).unsqueeze(-1)

        reidx = torch.tensor([self.reindex_dict[i] for i in range(self.n_genes)])
        gene_agg = torch.sparse.mm(interact_mat, entity_emb[reidx])
        input_emb = entity_agg[reidx]

        output_sl = []
        for i in range(self.n_factors):
            output_emb = GATConv[i](sl_graph, input_emb)
            output_sl.append(output_emb * latent_emb[i])

        output_sl = torch.stack(output_sl).permute(1, 0, 2)
        disen_weight = torch.mm(F.softmax(disen_weight_att, dim=-1), weight)
        disen_weight = disen_weight.expand(self.n_genes, self.n_factors, channel)

        gene_return = gene_agg + (output_sl * score).sum(dim=1)
        return entity_agg, gene_return, output_sl

### GraphConv Class

In [ ]:
class GraphConv(nn.Module):
    def __init__(self,
                 channel,
                 n_hops,
                 n_genes,
                 n_factors,
                 n_relations,
                 interact_mat,
                 ind,
                 reindex_dict,
                 node_dropout_rate=0.5,
                 mess_dropout_rate=0.1):
        super(GraphConv, self).__init__()
        self.GATConv = nn.ModuleList([
            GAT(1, channel, channel, channel, [8, 8], F.leaky_relu, 0.2, 0.2, 0.2, True)
            for _ in range(n_factors)
        ])

        self.interact_mat = interact_mat
        self.sl_graph = self.get_sl_graph()
        self.n_genes = n_genes
        self.n_factors = n_factors
        self.n_relations = n_relations
        self.reindex_dict = reindex_dict
        self.ind = ind
        self.node_dropout_rate = node_dropout_rate
        self.mess_dropout_rate = mess_dropout_rate

        self.weight = nn.Parameter(nn.init.xavier_uniform_(torch.empty(n_relations, channel)))
        self.disen_weight_att = nn.Parameter(nn.init.xavier_uniform_(torch.empty(n_factors, n_relations)))

        self.convs = nn.ModuleList([
            Aggregator(n_genes, n_factors, reindex_dict)
            for _ in range(n_hops)
        ])

        self.dropout = nn.Dropout(p=mess_dropout_rate)
        self.temperature = 0.2

    def get_sl_graph(self):
        indices = self.interact_mat._indices()
        g = dgl.graph((indices[0], indices[1]))
        return g.to(device)

    def _edge_sampling(self, edge_index, edge_type, rate=0.5):
        n_edges = edge_index.shape[1]
        idx = np.random.choice(n_edges, size=int(n_edges * rate), replace=False)
        return edge_index[:, idx], edge_type[idx]

    def _sparse_dropout(self, x, rate=0.5):
        noise_shape = x._nnz()
        random_tensor = rate + torch.rand(noise_shape, device=x.device)
        dropout_mask = torch.floor(random_tensor).bool()
        i = x._indices()[:, dropout_mask]
        v = x._values()[dropout_mask]
        return torch.sparse.FloatTensor(i, v, x.shape).to(x.device) * (1. / (1 - rate))

    def _cul_cor(self):
        def cosine_similarity(t1, t2):
            norm1 = t1 / t1.norm(dim=0, keepdim=True)
            norm2 = t2 / t2.norm(dim=0, keepdim=True)
            return (norm1 * norm2).sum(dim=0).pow(2)

        def mutual_information():
            T = self.disen_weight_att.t()
            T_norm = T / T.norm(dim=1, keepdim=True)
            pos_scores = torch.exp((T_norm * T_norm).sum(dim=1) / self.temperature)
            ttl_scores = torch.exp(torch.mm(T, self.disen_weight_att).sum(dim=1) / self.temperature)
            return -torch.sum(torch.log(pos_scores / ttl_scores))

        if self.ind == 'mi':
            return mutual_information()
        else:
            cor = 0
            for i in range(self.n_factors):
                for j in range(i + 1, self.n_factors):
                    if self.ind == 'distance':
                        cor += distance_correlation(self.disen_weight_att[i], self.disen_weight_att[j])
                    else:
                        cor += cosine_similarity(self.disen_weight_att[i], self.disen_weight_att[j])
            return cor

    def forward(self, gene_emb, entity_emb, latent_emb,
                edge_index, edge_type, interact_mat,
                mess_dropout=True, node_dropout=False):
        if node_dropout:
            edge_index, edge_type = self._edge_sampling(edge_index, edge_type, self.node_dropout_rate)
            interact_mat = self._sparse_dropout(interact_mat, self.node_dropout_rate)

        entity_res_emb = entity_emb
        gene_res_emb = gene_emb
        cor = self._cul_cor()
        for_reg = None

        for i, conv in enumerate(self.convs):
            entity_emb, gene_emb, output_gene = conv(
                self.GATConv, self.sl_graph, entity_emb, gene_emb, latent_emb,
                edge_index, edge_type, interact_mat, self.weight, self.disen_weight_att)

            if mess_dropout:
                entity_emb = self.dropout(entity_emb)
                gene_emb = self.dropout(gene_emb)

            entity_emb = F.normalize(entity_emb)
            gene_emb = F.normalize(gene_emb)

            entity_res_emb = entity_res_emb + entity_emb
            gene_res_emb = gene_res_emb + gene_emb

            mean_reg = output_gene.mean(dim=0)
            for_reg = mean_reg if for_reg is None else for_reg + mean_reg

        cor_2 = 0
        for i in range(self.n_factors):
            for j in range(i + 1, self.n_factors):
                cor_2 += distance_correlation(for_reg[i], for_reg[j])

        return entity_res_emb, gene_res_emb, cor, cor_2

### Define SLModel

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class SLModel(nn.Module):
    def __init__(self, n_genes, n_relations, n_entities, args_config, kg, sl_adj, reindex_dict):
        super(SLModel, self).__init__()
        self.n_genes = n_genes
        self.n_relations = n_relations
        self.n_entities = n_entities
        self.reindex_dict = {y: x for x, y in reindex_dict.items()}
        self.decay = args_config.l2
        self.sim_decay = args_config.sim_regularity
        self.emb_size = args_config.dim
        self.context_hops = args_config.context_hops
        self.n_factors = args_config.n_factors
        self.node_dropout = args_config.node_dropout
        self.node_dropout_rate = args_config.node_dropout_rate
        self.mess_dropout = args_config.mess_dropout
        self.mess_dropout_rate = args_config.mess_dropout_rate
        self.ind = args_config.ind
        self.device = torch.device(f"cuda:{args_config.gpu_id}" if args_config.cuda else "cpu")
        self.sl_adj = sl_adj
        self.kg = kg

        # Edge index and type for the knowledge graph
        self.edge_index, self.edge_type = self._get_edges(kg)

        # Initialize weights and GCN
        self._init_weight()
        self.all_embed = nn.Parameter(self.all_embed)
        self.latent_emb = nn.Parameter(self.latent_emb)
        self.gcn = self._init_model()

    def _init_weight(self):
        initializer = nn.init.xavier_uniform_
        self.all_embed = initializer(torch.empty(self.n_genes + self.n_entities, self.emb_size))
        self.latent_emb = initializer(torch.empty(self.n_factors, self.emb_size))
        self.interact_mat = self._convert_sp_mat_to_sp_tensor(self.sl_adj).to(self.device)

    def _init_model(self):
        return GraphConv(
            channel=self.emb_size,
            n_hops=self.context_hops,
            n_genes=self.n_genes,
            n_relations=self.n_relations,
            n_factors=self.n_factors,
            interact_mat=self.interact_mat,
            ind=self.ind,
            reindex_dict=self.reindex_dict,
            node_dropout_rate=self.node_dropout_rate,
            mess_dropout_rate=self.mess_dropout_rate
        )

    def _convert_sp_mat_to_sp_tensor(self, X):
        coo = X.tocoo()
        indices = torch.LongTensor(np.vstack((coo.row, coo.col)))
        values = torch.FloatTensor(coo.data)
        return torch.sparse.FloatTensor(indices, values, coo.shape)

    def _get_edges(self, graph):
        graph_tensor = torch.tensor(list(graph.edges))
        edge_index = graph_tensor[:, :-1].t().long().to(self.device)
        edge_type = graph_tensor[:, -1].long().to(self.device)
        return edge_index, edge_type

    def forward(self, gene_a, gene_b):
        gene_emb = self.all_embed[:self.n_genes]
        item_emb = self.all_embed[self.n_genes:]

        entity_gcn_emb, gene_gcn_emb, cor, cor_2 = self.gcn(
            gene_emb,
            item_emb,
            self.latent_emb,
            self.edge_index,
            self.edge_type,
            self.interact_mat,
            mess_dropout=self.mess_dropout,
            node_dropout=self.node_dropout
        )

        e_u = gene_gcn_emb[gene_a]

        reindexed = torch.tensor([self.reindex_dict[int(b)] for b in gene_b], device=self.device)
        e_e = entity_gcn_emb[reindexed]

        scores = (e_u * e_e).sum(dim=1)
        regularizer = (e_u.norm(2) ** 2 + e_e.norm(2) ** 2) / 2
        emb_loss = self.decay * regularizer / len(gene_a)
        cor_loss = self.sim_decay * (cor + cor_2)

        return torch.sigmoid(scores), emb_loss, cor_loss, cor

    def generate(self):
        gene_emb = self.all_embed[:self.n_genes]
        item_emb = self.all_embed[self.n_genes:]
        return self.gcn(
            gene_emb,
            item_emb,
            self.latent_emb,
            self.edge_index,
            self.edge_type,
            self.interact_mat,
            mess_dropout=False,
            node_dropout=False
        )[:-1]

    def rating(self, u_g_embeddings, i_g_embeddings):
        return torch.matmul(u_g_embeddings, i_g_embeddings.t())

    def create_bpr_loss(self, genes, pos_items, neg_items, cor):
        batch_size = genes.size(0)
        pos_scores = torch.sum(genes * pos_items, dim=1)
        neg_scores = torch.sum(genes * neg_items, dim=1)

        mf_loss = -torch.mean(nn.LogSigmoid()(pos_scores - neg_scores))

        regularizer = (genes.norm(2) ** 2 + pos_items.norm(2) ** 2 + neg_items.norm(2) ** 2) / 2
        emb_loss = self.decay * regularizer / batch_size
        cor_loss = self.sim_decay * cor

        return mf_loss + emb_loss + cor_loss, mf_loss, emb_loss, cor

**Question 1**: In the `GAT` class, there is a for loop that iteratively passes the hidden state `h` through `self.num_layers` of `GATConv`. Similarly, the `GraphConv` class iterates over the graph for `n_hops` using a list of `Aggregator` modules. What is the purpose of using multiple layers (or "hops") instead of just one? **(2.5 points)**

**Write answer here**

**Question 2**: The background describes two main operations: a Knowledge Graph (KG) Aggregation step, and a Factor-based Message Aggregation step. Look at the forward pass inside the Aggregator class and identify the two distinct message aggregation steps occurring in the code. Which variables/arguments dictate the graph structure for each of these two aggregations?  **(2.5 points)**

**Write answer here**:
1. Knowledge Graph Aggregation:
2. Factor/SL Aggregation:

**Question 3**: `GraphConv` initializes separate `GAT` instances for different `n_factors`. Why use multiple `GAT`s instead of a single standard GCN layer? **(2.5 points)**

**Write answer here**

## 2.3 <span style="color:blue">(2.5 points) </span> Instantiate SL Model

Here you will instantiate the model, move the model to the relevant device (cuda) as well as define the appropriate loss function (remember this is a classification task) and optimizer parameters. As in previous PSETs, you will be using the Adam optimizer to train the model. Once again, note the relevant hyperparameters have been defined in the **class ArgsConfig**.

In [ ]:
#this line of code should instantiate the model
model =
#this line of code should move the model to the relevant device (GPU)
model =
#this line of code should device the appropriate loss to be minimized during training
criterion =
#this line of code should define the optimization parameters (use Adam optimizer)
optimizer =

## 2.4 <span style="color:blue">(10 points) </span> Train & Evaluate Model (~8min if using A100)

Adapt the code shown in the Colab notebook to loop over the training and validation sets. Make sure to save the training and validation losses in the relevant lists and compute the relevant metrics during validation. Note that the loss being minimized during model training is a summed loss, including the Binary Cross-Entropy (BCE) loss (`torch.nn.BCELoss()`), a regularization (L2) loss, and a correlation loss based on a distance metric (the latter encourages the model to learn relationships between nodes that accurately reflect the graph structure). \\

During validation, we do not call the optimizer because the validation dataset is not used for training. You can make sure this is the case by calling `model.eval()` and turn on the `torch.no_grad()` context manager at the beginning of your validation function (we have implemented that for you). However, we need to call `model.train()` to turn the gradients and optimizer updates back during training. \\

Record the average train and validation loss for each epoch and plot these on a single graph. During validation, you will also compute the **Area Under the Curve (AUC), Precision, Recall, and F1 scores** using functions from sklearn.metrics. Finally, plot the precision vs recall in a single plot and comment on model performance.

Complete the loop shown below to train and evaluate the model. Compute the performance metrics using sklearn.metrics.

In [ ]:
training_loss_list, val_loss_list = [], []
auc_list, f1_list, precision_list, recall_list = [], [], [], []

for epoch in range(args_config.epoch):
    labels_train = torch.tensor([]).to(device)
    logits_train = torch.tensor([]).to(device)

    training_loss = 0

    model.train()

    for i, (gene_a, gene_b, labels) in enumerate(train_loader):
        gene_a = gene_a.to(device)
        gene_b = gene_b.to(device)
        labels = labels.to(device)

        ########## CODE ##########

        # Forward pass: pass gene_a and gene_b through the model
        # Get logits, emb_loss, cor_loss, _

        # Compute the total loss using the criterion + emb_loss + cor_loss

        # Backward pass: zero grad, loss backward, optimizer step

        # Accumulate training loss

        ########## CODE ##########

    training_loss = training_loss / len(train_loader)
    training_loss_list.append(training_loss)
    print(f"[Epoch {epoch}] Training Loss: {training_loss:.4f}")



    # ******** Validation set (we are now in evaluation mode) ********

    model.eval()
    with torch.no_grad():

        labels_val = torch.tensor([]).to(device)
        logits_val = torch.tensor([]).to(device)
        val_loss = 0

        for i, (gene_a, gene_b, labels) in enumerate(val_loader):
            gene_a = gene_a.to(device)
            gene_b = gene_b.to(device)
            labels = labels.to(device)

        ########## CODE ##########

            # Forward pass: model returns logits, emb_loss, cor_loss, _

            # Compute the total loss

            # Accumulate validation loss

        # Average validation loss for the epoch

        # Compute metrics

      ########## CODE ##########

### 2.5 <span style="color:blue">(5 points) </span> Plot Results (~1 min)

**There are three total tasks.**

**Task 1**: Plot the training and validation losses across epochs.

In [ ]:
########### CODE ###########

# plot the training and validation loss across epochs

########### CODE ###########

**Task 2**: Plot the performance metrics computed in the validation loop, including the Area under the Curve (AUC) and F1 score across epochs, and Precision vs Recall plots. (1.5 point)

In [ ]:
########### CODE ###########

# plot the AUC and F1 scores across epochs

########### CODE ###########

In [ ]:
########### CODE ###########

# plot the Precision vs. Recall

########### CODE ###########

**Task 3 Question**: Considering these plots, comment on model performance.

**Write answer here**

# 3. <span style="color:blue">(10 points) </span> Investigating one SL gene pair of interest

**This question requires no coding, just answer it in a markdown cell below.**

As hinted at in the background, the [SynLethDB Database](https://synlethdb.sist.shanghaitech.edu.cn/#/search) contains information about human SL gene pairs, most of them relevant in human cancers. One of the gene pairs in this study includes HOXC11, a transcription factor involved in lung and colorectal cancer, and KRAS, a well-known oncogene (i.e., a gene that underwent mutations promoting unregulated cell growth and division).

**Question**: Look up the entry for HOXC11 in the [SynLethDB Database](https://synlethdb.sist.shanghaitech.edu.cn/#/search) -  what SL interactions does it establish with other genes? Would you expect KRAS inhibitors to be particularly effective in cancer cells bearing loss of function mutations in HOXC11? Explain your answer.

**Write answer here**

---

# Submission

Congratulations! You've reached the end of the pset. Please submit your completed work as a `.ipynb` to Gradescope. Furthermore, if you had any AI-based assistance or worked with collaborators, please list them in the following cell.

**For submission:**
Please take a look at the `# --- Configuration ---` section of the next cell. You will need to change the `NOTEBOOK_NAME` to match the name of your google colab notebook. After making the applicable changes, please run the cell which will reduce the size of your generated images and aim for a file size of less than 10 mb. Note that it will also delete cells tagged as "background", but you don't need to worry about it deleting your own code.

When successful, you'll be prompted to download the reformatted notebook which you can then upload to gradescope.

In [ ]:
import io
import os
import base64
import nbformat
import sys
from PIL import Image
from IPython.display import display, Javascript, HTML

# --- Configuration ---
NOTEBOOK_NAME = "ASSIGNMENT_NAME.ipynb" # The name of your file
OUTPUT_FILENAME = "submission.ipynb"
TAG_TO_REMOVE = "background"
MAX_IMG_WIDTH = 800
# Standard locations where Colab saves notebooks.
# IF YOU CHANGE THE LOCATION OF YOUR NOTEBOOK PLEASE ADD THE PATH HERE.
COLAB_PATHS = [
    f"/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_NAME}",
    f"/content/drive/MyDrive/{NOTEBOOK_NAME}"
]
# ---------------------

def get_input_path():
    """Determines the path of the notebook based on the environment."""
    if 'google.colab' in sys.modules:
        from google.colab import drive
        # 1. Mount Drive
        if not os.path.exists('/content/drive'):
            print("Mounting Google Drive to access the notebook file...")
            drive.mount('/content/drive')

        for path in COLAB_PATHS:
            if os.path.exists(path):
                return path

        # Fallback if not found
        print(f"\nERROR: Could not find '{NOTEBOOK_NAME}' in your Google Drive.")
        print("Please ensure the file is saved in 'My Drive' or 'Colab Notebooks'.")
        return None

    else:
        # Local Jupyter (runs in current directory)
        return NOTEBOOK_NAME

def resize_base64_image(b64_str, mime_type):
    # (Same resize logic as before - keeping it brief for readability)
    try:
        img_data = base64.b64decode(b64_str)
        img = Image.open(io.BytesIO(img_data))
        if img.width > MAX_IMG_WIDTH:
            ratio = MAX_IMG_WIDTH / img.width
            new_height = int(img.height * ratio)
            img = img.resize((MAX_IMG_WIDTH, new_height), Image.Resampling.LANCZOS)
            buf = io.BytesIO()
            fmt = 'PNG' if 'png' in mime_type else 'JPEG'
            img.save(buf, format=fmt, optimize=True)
            return base64.b64encode(buf.getvalue()).decode('utf-8')
        return b64_str
    except Exception as e:
        return b64_str

def generate_submission():
    # Trigger a save in the browser
    display(Javascript('IPython.notebook.save_checkpoint();'))

    input_path = get_input_path()
    if not input_path:
        return

    print(f"Reading notebook from: {input_path}")

    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            nb = nbformat.read(f, as_version=4)
    except Exception as e:
        print(f"Error reading file: {e}")
        return

    new_cells = []

    # Filter Cells & Process Images
    for cell in nb.cells:
        tags = cell.get('metadata', {}).get('tags', [])
        tags = [t.lower() for t in tags] if tags else []
        if TAG_TO_REMOVE in tags:
            continue

        if 'outputs' in cell:
            for output in cell['outputs']:
                data = output.get('data', {})
                for mime_type in ['image/png', 'image/jpeg']:
                    if mime_type in data:
                        data[mime_type] = resize_base64_image(data[mime_type], mime_type)
        new_cells.append(cell)

    nb.cells = new_cells

    with open(OUTPUT_FILENAME, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)

    # Download logic for Colab
    if 'google.colab' in sys.modules:
        from google.colab import files
        print(f"Downloading {OUTPUT_FILENAME}...")
        files.download(OUTPUT_FILENAME)
        print(f"Success! {OUTPUT_FILENAME} downloaded.")
    else:
        print(f"Success! {OUTPUT_FILENAME} created.")
        display(HTML(f'<br/><a href="{OUTPUT_FILENAME}" download><b>Click here to download {OUTPUT_FILENAME}</b></a>'))

generate_submission()